# Column Transformer 
A Column Transformer is a preprocessing utility in scikit-learn that allows you to apply different transformations to different columns of a dataset simultaneously.

## Key Idea
Real-world datasets usually contain different types of features:
- Numerical features → need scaling or normalization
- Categorical features → need encoding

Instead of preprocessing them separately, ColumnTransformer handles everything in one pipeline.

In [40]:
import numpy as np
import pandas as pd

In [41]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [42]:
df =pd.read_csv("https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day28-column-transformer/covid_toy.csv")

In [43]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [44]:
df.isnull().sum() # tio check the missing values 

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [45]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],
                                                test_size=0.2)

In [46]:
X_train

,age,gender,fever,cough,city
5,84,Female,NaN,Mild,Bangalore
42,27,Male,100.0,Mild,Delhi
38,49,Female,101.0,Mild,Delhi
73,34,Male,98.0,Strong,Kolkata
76,80,Male,100.0,Mild,Bangalore
...,...,...,...,...,...
13,64,Male,102.0,Mild,Bangalore
44,20,Male,102.0,Strong,Delhi
50,19,Male,101.0,Mild,Delhi
21,73,Male,98.0,Mild,Bangalore


## Normal Method 

In [47]:
# adding simple imputer to fever col
si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])
                                 
X_train_fever.shape

(80, 1)

In [48]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data
X_test_cough = oe.fit_transform(X_test[['cough']])

X_train_cough.shape

(80, 1)

In [49]:
from sklearn.preprocessing import OneHotEncoder

# create encoder
ohe = OneHotEncoder(drop='first', sparse_output=False)

# fit only on training data
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# transform test data using same encoder
X_test_gender_city = ohe.transform(X_test[['gender','city']])

# check shape
X_train_gender_city.shape

(80, 4)

In [50]:
# Extracting Age
X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values

X_train_age.shape

(80, 1)

In [51]:
X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1) # type: ignore

X_train_transformed.shape

(80, 7)

# Column Transformer in SK learn method 

In [52]:
from sklearn.compose import ColumnTransformer

In [53]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

transformer = ColumnTransformer(transformers=[
    ('tnf1', SimpleImputer(), ['fever']),
    ('tnf2', OrdinalEncoder(categories=[['Mild','Strong']]), ['cough']),
    ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender','city'])
], remainder='passthrough')

In [54]:
transformer.fit_transform(X_train).shape

(80, 7)

In [55]:
transformer.transform(X_test).shape

(20, 7)